<a href="https://colab.research.google.com/github/doha-RISSE/speech-corpus-for-low-ressource-langages/blob/notebooks/FastText.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets --quiet

from datasets import load_dataset
import re

# Charger le dataset ArabicSpeech/MGB-5 (train split)
dataset_mgb5 = load_dataset("ArabicSpeech/MGB-5", split="train")

print("Nombre d'exemples dans MGB-5 :", dataset_mgb5.num_rows)

# Fonction pour nettoyer une phrase arabe
def clean_arabic_text(text):
    # 1) normaliser différentes formes d'alifs
    text = re.sub(r"[إأآٱ]", "ا", text)

    # 2) supprimer les chiffres arabes et latins
    text = re.sub(r"[0-9٠١٢٣٤٥٦٧٨٩]", " ", text)

    # 3) supprimer la ponctuation arabe et latine et les symboles
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)

    # 4) réduire les espaces multiples
    text = re.sub(r"\s+", " ", text).strip()

    return text

# Ajouter les transcriptions au fichier global
with open("darija_corpus.txt", "a", encoding="utf-8") as f:
    for example in dataset_mgb5:
        # Récupérer la colonne "text"
        if "text" in example and example["text"] is not None:
            txt = example["text"]
            # Vérifier que c'est bien une chaîne
            if isinstance(txt, str):
                # Nettoyer le texte
                text_clean = clean_arabic_text(txt)

                # Écrire seulement s'il reste du texte après nettoyage
                if text_clean:
                    f.write(text_clean + "\n")

print("✔️ Transcriptions nettoyées et ajoutées depuis ArabicSpeech/MGB-5 dans darija_corpus.txt")

Nombre d'exemples dans MGB-5 : 33338
✔️ Transcriptions nettoyées et ajoutées depuis ArabicSpeech/MGB-5 dans darija_corpus.txt


# entrainer model

In [ ]:
# ============================================================
# CELLULE 1 — Uploader le fichier proprement sur Colab
# ============================================================
from google.colab import files

uploaded = files.upload()  # ← clique sur "Choisir un fichier"
                           #   et sélectionne ton .parquet

Saving ner_annotated_all.parquet to ner_annotated_all.parquet


In [ ]:
# ============================================================
# CELLULE 1 — Installation
# ============================================================
!pip install gensim pyarrow --quiet

In [ ]:
# ============================================================
# CELLULE 3 — CODE COMPLET (le fichier est déjà uploadé)
# ============================================================
import os
import re
import unicodedata
import pyarrow.parquet as pq
from gensim.models import FastText
from gensim.models.callbacks import CallbackAny2Vec

# Récupérer le nom du fichier uploadé dans la cellule précédente
filename = list(uploaded.keys())[0]
print(f"Fichier : {filename}  ({os.path.getsize(filename)/1024/1024:.1f} MB)")

# Vérification
with open(filename, "rb") as f:
    header = f.read(4)
    f.seek(-4, 2)
    footer = f.read(4)
assert header == b"PAR1" and footer == b"PAR1", "❌ Fichier corrompu — re-uploader"
print("✅ Fichier parquet valide")

# ── 1) Nettoyage ──────────────────────────────────────────
def clean_sentence(text):
    if not isinstance(text, str):
        return []
    text = re.sub(r"http\S+|www\.\S+", "", text)
    cleaned = [ch for ch in text
               if unicodedata.category(ch).startswith(("L","N","Z","P"))
               or ch in "،,.:;!?-()"]
    text = re.sub(r"\s+", " ", "".join(cleaned)).strip()
    return text.split()

# ── 2) Lecture parquet → phrases uniques ─────────────────
pf = pq.ParquetFile(filename)
print(f"Row groups : {pf.num_row_groups}")

seen_ids, sentences = set(), []

for i in range(pf.num_row_groups):
    batch = pf.read_row_group(i, columns=["id", "sentence"])
    df    = batch.to_pandas().drop_duplicates(subset="id")
    for _, row in df.iterrows():
        sid = row["id"]
        if sid not in seen_ids:
            seen_ids.add(sid)
            tokens = clean_sentence(row["sentence"])
            if len(tokens) >= 2:
                sentences.append(tokens)
    if (i + 1) % 10 == 0:
        print(f"  [{i+1}/{pf.num_row_groups}] phrases : {len(sentences):,}")

print(f"\n✅ Total phrases uniques : {len(sentences):,}")

# ── 3) Entraînement FastText ──────────────────────────────
class LossLogger(CallbackAny2Vec):
    def __init__(self): self.epoch = 1
    def on_epoch_end(self, model):
        print(f"  Epoch {self.epoch:02d} — loss : {model.get_latest_training_loss():.2f}")
        self.epoch += 1

print("\nEntraînement FastText...")
model = FastText(
    sentences   = sentences,
    vector_size = 300,
    window      = 5,
    min_count   = 3,
    sg          = 1,
    hs          = 0,
    negative    = 10,
    min_n       = 2,
    max_n       = 5,
    epochs      = 15,
    alpha       = 0.05,
    min_alpha   = 0.0001,
    workers     = 4,
    seed        = 42,
    callbacks   = [LossLogger()]
)
print(f"\n✅ Vocabulaire : {len(model.wv.key_to_index):,} mots")

# ── 4) Sauvegarde ─────────────────────────────────────────
model.save("darija_fasttext.model")
model.wv.save_word2vec_format("darija_fasttext_vectors.vec")
print("✅ darija_fasttext.model       sauvegardé")
print("✅ darija_fasttext_vectors.vec sauvegardé")

# ── 5) Tests de similarité ────────────────────────────────
print("\n── Tests ──")
for word in ["مزيان", "كيفاش", "بغيت", "دار", "ماكاين"]:
    try:
        sims = model.wv.most_similar(word, topn=5)
        print(f"\n'{word}' → {[w for w,_ in sims]}")
    except KeyError:
        print(f"\n'{word}' → absent du vocabulaire")

Fichier : ner_annotated_all.parquet  (72.2 MB)
✅ Fichier parquet valide
Row groups : 62
  [10/62] phrases : 36,898
  [20/62] phrases : 75,113
  [30/62] phrases : 126,361
  [40/62] phrases : 176,201
  [50/62] phrases : 228,124
  [60/62] phrases : 262,873

✅ Total phrases uniques : 269,399

Entraînement FastText...
  Epoch 01 — loss : 0.00
  Epoch 02 — loss : 0.00
  Epoch 03 — loss : 0.00
  Epoch 04 — loss : 0.00
  Epoch 05 — loss : 0.00
  Epoch 06 — loss : 0.00
  Epoch 07 — loss : 0.00
  Epoch 08 — loss : 0.00
  Epoch 09 — loss : 0.00
  Epoch 10 — loss : 0.00
  Epoch 11 — loss : 0.00
  Epoch 12 — loss : 0.00
  Epoch 13 — loss : 0.00
  Epoch 14 — loss : 0.00
  Epoch 15 — loss : 0.00

✅ Vocabulaire : 180,162 mots
✅ darija_fasttext.model       sauvegardé
✅ darija_fasttext_vectors.vec sauvegardé

── Tests ──

'مزيان' → ['مزيان،', 'مزيان.', 'مزيان؛', 'مزيانه', 'ومزيان']

'كيفاش' → ['وكيفاش', 'لكيفاش', 'فكيفاش', '"كيفاش', 'بكيفاش']

'بغيت' → ['وبغيت', 'فبغيت', 'مابغيت', 'خليني', 'نبغي']

'دار

In [ ]:
for word in ["اللي", "اواااه", "معمرني", "هادشي", "ماتقولش"]:
    try:
        sims = model.wv.most_similar(word, topn=5)
        print(f"\n'{word}' → {[w for w,_ in sims]}")
    except KeyError:
        print(f"\n'{word}' → absent du vocabulaire")


'اللي' → ['لي', 'واللي', 'ديال', 'ولي', 'ما']

'اواااه' → ['ااااه', 'اوا', 'شحااال', 'اوائل', 'اوف']

'معمرني' → ['عمرني', 'وعمرني', 'معمرنا', 'معمرها', 'ماعمرني']

'هادشي' → ['وهادشي', 'داكشي', 'هاداكشي', 'وهاداكشي', '-وهادشي']

'ماتقولش' → ['ماتقول', 'ماتقولو', 'ماتقولوش', 'ماتقوليش', 'تقولش']


In [ ]:
# ============================================================
# Similarités pour tous les mots uniques → fichier TSV
# ============================================================
import csv
from google.colab import files

# ── 1) Extraire tous les mots uniques des phrases ─────────
print("Extraction des mots uniques...")

all_words = set()
with open("phrases.txt", "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("--- ID:"):
            continue
        for word in line.split():
            all_words.add(word)

print(f"✅ Mots uniques trouvés : {len(all_words):,}")

# ── 2) Calculer les similarités et écrire le fichier ──────
output_file = "similarites_darija.tsv"
absent = 0
traites = 0

with open(output_file, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerow(["mot", "similaire_1", "score_1",
                              "similaire_2", "score_2",
                              "similaire_3", "score_3",
                              "similaire_4", "score_4",
                              "similaire_5", "score_5"])

    for word in sorted(all_words):
        try:
            sims = model.wv.most_similar(word, topn=5)
            row  = [word]
            for w, score in sims:
                row += [w, f"{score:.4f}"]
            writer.writerow(row)
            traites += 1
        except KeyError:
            absent += 1

        if (traites + absent) % 10000 == 0:
            print(f"  Traités : {traites:,}  |  Absents : {absent:,}")

print(f"\n✅ Terminé !")
print(f"   Mots traités  : {traites:,}")
print(f"   Mots absents  : {absent:,}")
print(f"   Fichier       : {output_file}")

# ── 3) Téléchargement automatique ─────────────────────────
files.download(output_file)

Extraction des mots uniques...
✅ Mots uniques trouvés : 367,521
  Traités : 10,000  |  Absents : 0
  Traités : 20,000  |  Absents : 0
  Traités : 30,000  |  Absents : 0
  Traités : 40,000  |  Absents : 0
  Traités : 50,000  |  Absents : 0
  Traités : 60,000  |  Absents : 0
  Traités : 70,000  |  Absents : 0
  Traités : 80,000  |  Absents : 0
  Traités : 90,000  |  Absents : 0
  Traités : 100,000  |  Absents : 0
  Traités : 110,000  |  Absents : 0
  Traités : 120,000  |  Absents : 0
  Traités : 130,000  |  Absents : 0
  Traités : 140,000  |  Absents : 0
  Traités : 150,000  |  Absents : 0
  Traités : 160,000  |  Absents : 0
  Traités : 170,000  |  Absents : 0
  Traités : 180,000  |  Absents : 0
  Traités : 190,000  |  Absents : 0
  Traités : 200,000  |  Absents : 0
  Traités : 210,000  |  Absents : 0
  Traités : 220,000  |  Absents : 0
  Traités : 230,000  |  Absents : 0
  Traités : 240,000  |  Absents : 0
  Traités : 250,000  |  Absents : 0
  Traités : 260,000  |  Absents : 0
  Traités

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install gensim --quiet

from gensim.models import FastText

# ------------------------------------
# 1) Préparer les données du corpus
# ------------------------------------
sentences = []

with open("darija_corpus (2).txt", "r", encoding="utf-8") as f:
    for line in f:
        # chaque ligne est une phrase propre, on split par espaces
        tokens = line.strip().split()
        if tokens:            # ignorer les lignes vides
            sentences.append(tokens)

print(f" Nombre de phrases chargées : {len(sentences)}")

# ------------------------------------
# 2) Entraîner le modèle FastText
# ------------------------------------
# FastText est adapté aux langues avec beaucoup de variations ou formes rares
# car il utilise des sous‑mots (caractères n‑grams). :contentReference[oaicite:1]{index=1}

model = FastText(
    sentences=sentences,
    vector_size=100,   # taille des vecteurs (100 est un bon point de départ)
    window=5,          # taille du contexte
    min_count=5,       # ignorer les mots fréquents <5
    epochs=10,         # nombre d'itérations d'entraînement
    sg=1,              # 1 => Skip‑gram, souvent meilleur pour embeddings
)

# ------------------------------------
# 3) Sauvegarder le modèle
# ------------------------------------
model.save("darija_fasttext.model")
model.wv.save_word2vec_format("darija_fasttext_vectors.vec")

print(" Modèle FastText entraîné et sauvegardé !")
print(" Vocabulaire appris :", len(model.wv.key_to_index))

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xd9 in position 0: unexpected end of data

In [ ]:
similar = model.wv.most_similar("بغييت", topn=10)
print(similar)

[('بغيتش', 0.8955931663513184), ('شبغيت', 0.8870959877967834), ('بغيت', 0.8800671100616455), ('لبغيت', 0.8601251244544983), ('وبغيت', 0.8278326988220215), ('مبغيت', 0.8268101811408997), ('بغي', 0.8192673325538635), ('بغيتيني', 0.8188920021057129), ('بغيتكم', 0.8024687170982361), ('بغيتك', 0.7888990044593811)]


In [ ]:
similar = model.wv.most_similar("اواااه", topn=10)
print(similar)

[('اواه', 0.9356077313423157), ('اوا', 0.8332635164260864), ('لاواه', 0.8307888507843018), ('سحاره', 0.8183757066726685), ('اوت', 0.8164303302764893), ('اوف', 0.8096382021903992), ('ياه', 0.8094990253448486), ('حاره', 0.8042316436767578), ('اعشيري', 0.7999936938285828), ('اصحبتي', 0.7964184284210205)]


In [ ]:
similar = model.wv.most_similar("لمغريب", topn=10)
print(similar)

[('تدريب', 0.7998566031455994), ('المغريبي', 0.7914615273475647), ('قزح', 0.777472734451294), ('توفر', 0.7716092467308044), ('تطوان', 0.7689029574394226), ('غريب', 0.7654107809066772), ('مقتانع', 0.7608031034469604), ('مهتم', 0.7591993808746338), ('محظوظ', 0.7585315704345703), ('فخط', 0.7568997740745544)]


In [ ]:
!pip install python-Levenshtein
from Levenshtein import distance

valid_variants = [w for w, score in similar if distance(w, "لمغريب") <= 2]
print(valid_variants)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 79.2 MB/s eta 0:00:00
['المغريبي', 'غريب']


In [ ]:
similar = model.wv.most_similar("هادشي", topn=20)
print(similar)

[('لهادشي', 0.9093756079673767), ('بهادشي', 0.9021050333976746), ('وهادشي', 0.8880813717842102), ('فهادشي', 0.8740510940551758), ('وهدشي', 0.8221244812011719), ('لهدشي', 0.8079528212547302), ('فهدشي', 0.7695450186729431), ('هدشي', 0.7630062103271484), ('هادكشي', 0.7592264413833618), ('ديكشي', 0.6751107573509216), ('اوداكشي', 0.6726335287094116), ('هدكشي', 0.6589253544807434), ('لداكشي', 0.6518052816390991), ('هداكشي', 0.6513252854347229), ('وهداكشي', 0.6496257185935974), ('داكشي', 0.6474707126617432), ('فداكشي', 0.6431254148483276), ('وداكشي', 0.6367235779762268), ('دكشي', 0.6360898017883301), ('بداكشي', 0.6307345032691956)]


In [ ]:
!pip install levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 45.2 MB/s eta 0:00:00


In [ ]:
from Levenshtein import distance

valid_variants = [w for w, score in similar if distance(w, "هادشي") <= 2]
print(valid_variants)

['لهادشي', 'بهادشي', 'وهادشي', 'فهادشي', 'وهدشي', 'لهدشي', 'فهدشي', 'هدشي', 'هادكشي', 'هدكشي', 'هداكشي', 'داكشي']


In [ ]:
similar = model.wv.most_similar("الماء", topn=10)
print(similar)

[('الما', 0.8635035753250122), ('الماطش', 0.86173015832901), ('الماضي', 0.851513683795929), ('الماعن', 0.8508082628250122), ('المالح', 0.8483149409294128), ('المالكي', 0.8451757431030273), ('المازوط', 0.8394877910614014), ('المانط', 0.8338415026664734), ('الماستر', 0.8325265049934387), ('المال', 0.8291005492210388)]


In [ ]:
similar = model.wv.most_similar("ماتقولش", topn=20)
print(similar)

[('متقولش', 0.8763700723648071), ('متقوليش', 0.8122743368148804), ('غتقول', 0.8082416653633118), ('لفكد', 0.8060433864593506), ('بزقول', 0.8056581020355225), ('ماجد', 0.8044672012329102), ('ماشغلكش', 0.8032583594322205), ('كتقوليه', 0.802837073802948), ('وتقول', 0.7982111573219299), ('ماحشمتيش', 0.7966702580451965), ('غتقولي', 0.796370267868042), ('بّاك', 0.7957143187522888), ('ماتت', 0.7942163348197937), ('ماتخافش', 0.7906111478805542), ('شغنقولك', 0.7892670035362244), ('شغلك', 0.7889003157615662), ('متكولش', 0.7874149084091187), ('دفعتيني', 0.7861599326133728), ('مسمعتش', 0.7856964468955994), ('ماشتيش', 0.784532904624939)]


In [ ]:
from Levenshtein import distance

valid_variants = [w for w, score in similar if distance(w, "ماتقولش") <= 2]
print(valid_variants)

['متقولش', 'متقوليش', 'متكولش']


In [ ]:
similar = model.wv.most_similar("عشى", topn=20)
print(similar)

[('عشا', 0.8304281830787659), ('جويي', 0.8098196387290955), ('عشرا', 0.8084096312522888), ('عشر', 0.8082160949707031), ('عشره', 0.8023532032966614), ('جلست', 0.7978994250297546), ('نيويورك', 0.786148190498352), ('فايق', 0.7860404253005981), ('تلفازة', 0.7799249887466431), ('شايط', 0.7773428559303284), ('قرعه', 0.7729383707046509), ('تمشى', 0.7671517729759216), ('حبست', 0.7651439309120178), ('نربح', 0.7644801735877991), ('لكندا', 0.7633890509605408), ('نفترضو', 0.7632574439048767), ('ورجعت', 0.76283860206604), ('يتمشى', 0.7619074583053589), ('لابيسين', 0.7609903812408447), ('مضببه', 0.7608774304389954)]


In [ ]:
from Levenshtein import distance

valid_variants = [w for w, score in similar if distance(w, "عشى ") <= 2]
print(valid_variants)

['عشا', 'عشرا', 'عشر', 'عشره']


In [ ]:
!pip install datasets --quiet

from datasets import load_dataset
import re

# Charger le dataset wikitongues-darija
dataset = load_dataset("BrunoHays/wikitongues-darija", split="test")

print("Nombre d'exemples :", dataset.num_rows)

# Fonction nettoyage (en gardant les chiffres)
def clean_arabic_text(text):
    # 1) normaliser différentes formes d'alifs
    text = re.sub(r"[إأآٱ]", "ا", text)

    # 2) supprimer caractères non-arabes sauf les chiffres
    text = re.sub(r"[^\u0600-\u06FF0-9٠١٢٣٤٥٦٧٨٩\s]", " ", text)

    # 3) supprimer espaces multiples
    text = re.sub(r"\s+", " ", text).strip()

    return text

# Liste pour stocker les transcriptions nettoyées
cleaned_transcriptions = []

for example in dataset:
    if "transcription" in example and example["transcription"]:
        txt = example["transcription"]
        if isinstance(txt, str):
            text_clean = clean_arabic_text(txt)
            if text_clean:
                cleaned_transcriptions.append(text_clean)

print("✔️ Transcriptions nettoyées prêtes")
print(cleaned_transcriptions)

README.md:   0%|          | 0.00/923 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/42 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/42 [00:00<?, ?it/s]

data/test/anass-105-00-120-00.wav:   0%|          | 0.00/480k [00:00<?, ?B/s]

data/test/anass-134-00-151-00.wav:   0%|          | 0.00/544k [00:01<?, ?B/s]

...

data/test/nawal-92-34-97-10.wav:   0%|          | 0.00/152k [00:00<?, ?B/s]

data/test/nawal-57-69-65-26.wav:   0%|          | 0.00/242k [00:00<?, ?B/s]

data/test/nawal-98-34-103-86.wav:   0%|          | 0.00/177k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/41 [00:00<?, ? examples/s]

Nombre d'exemples : 41
✔️ Transcriptions nettoyées prêtes
['السلام عليكم، انا سميتي نوال عمري 33 عام كنخدم استاذة ديال الاجانب كنقريهم الدارجة والعربية الفصحى كيجيو بزاف ديال الطلبة من العالم كامل من فرنسا، من ايطاليا، من اسيا، من ميريكان، من كندا، من البلايص كاملين فالعالم', 'او كيجيو باش يتعلمو شوية العربية الفصحى، او كاين لي كيبغي يتعلم حتا الدارجة', 'كاين الطلبة لي كيبغيو يتعلمو حيت كيقراو العربية فالجامعة ديالهم', 'او كاين الطلبة لي كيبغيو غير يتعلمو العربية، حيت كيجيو هنا فالمغرب سياحة', 'كيجيو يشوفو المغرب والمدن المغربية او فنفس الوقت كيجيو يتعلمو شوية ديال الدارجة باش يقدرو يتكلمو شوية مع الناس فالزنقة، فطاكسي، فريسطو فاي بلاصة', 'المغرب بلاد زوينة بزاف دالناس كيجيو ليها حيت الجو ديالها زوين كاين عندنا الطبيعة مختلفة', 'كاين الجبل، كاين البحر، كاين كاين الصحرا او الجو ديما زوين، سخون او بزاف دالاجانب كيعجبهم الحال هنا فالمغرب حيت الجو مشمش او زوين بزاف', 'التجربة ديالي فهاد الخدمة زوينة بزاف كنخدم هادي تقريبا ست سنين او نص', 'او عندي الكونطاكط زوين مع الطلبة', 'سيغتو هما باقين

# Conversion de chiffres en darija code dial gemini


In [ ]:
import re

def nombre_vers_darija(n):
    try:
        n = int(n)
    except ValueError:
        return str(n)

    # Dictionnaires de base pour la Darija
    unites = ["", "واحد", "جوج", "تلاتة", "ربعة", "خمسة", "ستة", "سبعة", "تمنية", "تسعود"]
    dix_a_dix_neuf = ["عشرة", "حداش", "طناش", "تلطاش", "ربعطاش", "خمسطاش", "سطاش", "سبعطاش", "تمنطاش", "تسعطاش"]
    dizaines = ["", "عشرة", "عشرين", "تلاتين", "ربعين", "خمسين", "ستين", "سبعين", "تمنين", "تسعين"]
    centaines = ["", "مية", "ميتين", "تلت مية", "ربع مية", "خمس مية", "ست مية", "سبع مية", "تمن مية", "تسع مية"]

    if n == 0:
        return "زيرو" # ou "صفر"

    def convertir_moins_de_100(num):
        if num < 10:
            return unites[num]
        elif 10 <= num < 20:
            return dix_a_dix_neuf[num - 10]
        else:
            t = num // 10
            u = num % 10
            if u == 0:
                return dizaines[t]
            else:
                return unites[u] + " و" + dizaines[t]

    if n < 100:
        return convertir_moins_de_100(n)
    elif n < 1000:
        h = n // 100
        rem = n % 100
        if rem == 0:
            return centaines[h]
        else:
            return centaines[h] + " و" + convertir_moins_de_100(rem)
    else:
        # Si le nombre est >= 1000, on le laisse en chiffre pour l'instant
        return str(n)

def transformer_transcription(texte):
    # Cherche tous les nombres (composés de chiffres) et les remplace
    return re.sub(r'\b\d+\b', lambda match: nombre_vers_darija(match.group()), texte)

# 1. Ta liste de transcriptions
transcriptions = [
    'السلام عليكم، انا سميتي نوال عمري 33 عام...', # (Tu peux coller toute ta liste ici)
    'السلام عليكم انا سميتي اناس عندي 28 عام و كانصور هاد الفيديو د الدارجة المغربية هنايا فالسويد',
    'قبائل بني هلال هوما واحد القبائل عربية لي جاو فالقرن 13 الا بقيت عاقل مزيان للمغرب',
    'كاين تقريبا شي 642 لهجة ديال الامازيغية'
]

# 2. Application de la transformation
transcriptions_propres = [transformer_transcription(t) for t in transcriptions]

# 3. Affichage du résultat
for t in transcriptions_propres:
    print(t)

السلام عليكم، انا سميتي نوال عمري تلاتة وتلاتين عام...
السلام عليكم انا سميتي اناس عندي تمنية وعشرين عام و كانصور هاد الفيديو د الدارجة المغربية هنايا فالسويد
قبائل بني هلال هوما واحد القبائل عربية لي جاو فالقرن تلطاش الا بقيت عاقل مزيان للمغرب
كاين تقريبا شي ست مية وجوج وربعين لهجة ديال الامازيغية


# TN

In [ ]:
!pip install nemo_text_processing

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 69.3 MB/s eta 0:00:00
  Created wheel for cdifflib: filename=cdifflib-1.2.9-cp312-cp312-linux_x86_64.whl size=29758 sha256=0d2eab24062a756d53866a4869e380c4ee25fe59447bae4862a2f5a6f7912826
  Stored in directory: /root/.cache/pip/wheels/4b/26/dc/0c60f17cb2fee90ffc80231b11cd034572cd05c64711a74534
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9655 sha256=f1d4bc5e2596dbda75462e6a20cd639a1f824199ba566833a62640fe4abccfc6
  Stored in directory: /root/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built cdifflib wget


In [ ]:
!python -m nemo_text_processing.text_normalization.normalize \
  --text "123" \
  --language ar

/usr/bin/python3: Error while finding module specification for 'nemo_text_processing.text_normalization.normalize' (ModuleNotFoundError: No module named 'nemo_text_processing')


In [ ]:
from nemo_text_processing.text_normalization.normalize import Normalizer

# Assure toi d’avoir installé nemo_text_processing
# pip install nemo-text-processing

# Si ton texte contient des lettres majuscules, utilise "cased"
# sinon utilise "lower_cased"
normalizer = Normalizer(
    input_case="lower_cased",  # ou "cased"
    lang="ar",                 # pour l’arabe
    cache_dir="cache"
)

print(normalizer.normalize("123"))
print(normalizer.normalize("45 MAD"))
print(normalizer.normalize("75%"))
print(normalizer.normalize("12/05/2024"))
print(normalizer.normalize("03:30"))

 NeMo-text-processing :: INFO     :: Creating ClassifyFst grammars. This might take some time...
INFO:NeMo-text-processing:Creating ClassifyFst grammars. This might take some time...
 NeMo-text-processing :: INFO     :: Created cache/_lower_cased_ar_tn_True_deterministic.far
INFO:NeMo-text-processing:Created cache/_lower_cased_ar_tn_True_deterministic.far
 NeMo-text-processing :: INFO     :: Created cache/ar_tn_True_deterministic_verbalizer.far
INFO:NeMo-text-processing:Created cache/ar_tn_True_deterministic_verbalizer.far


مئة وثلاثة وعشرون
خمسة وأربعون MAD
خمسة وسبعون في المائة
12/05/2024
03:30


In [ ]:
!apt-get install -y libfst-dev
!pip install pynini==2.1.5
!pip install nemo_text_processing

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libfst8
The following NEW packages will be installed:
  libfst-dev libfst8
0 upgraded, 2 newly installed, 0 to remove and 45 not upgraded.
Need to get 7,452 kB of archives.
After this operation, 113 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libfst8 amd64 1.6.3-2ubuntu1 [2,767 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libfst-dev amd64 1.6.3-2ubuntu1 [4,685 kB]
Fetched 7,452 kB in 2s (3,775 kB/s)
Selecting previously unselected package libfst8.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../libfst8_1.6.3-2ubuntu1_amd64.deb ...
Unpacking libfst8 (1.6.3-2ubuntu1) ...
Selecting previously unselected package libfst-dev.
Preparing to unpack .../libfst-dev_1.6.3-2ubuntu1_amd64.deb ...
Unpacking libfst-dev (1.6.3-2u

In [ ]:
from nemo_text_processing.text_normalization.normalize import Normalizer

normalizer = Normalizer(
    input_case="lower_cased",
    lang="ar",
    cache_dir="cache"
)

# IMPORTANT : debug classification
print(normalizer.normalize("123", verbose=True))

 NeMo-text-processing :: INFO     :: Creating ClassifyFst grammars. This might take some time...
INFO:NeMo-text-processing:Creating ClassifyFst grammars. This might take some time...
 NeMo-text-processing :: INFO     :: Created cache/_lower_cased_ar_tn_True_deterministic.far
INFO:NeMo-text-processing:Created cache/_lower_cased_ar_tn_True_deterministic.far
 NeMo-text-processing :: INFO     :: Created cache/ar_tn_True_deterministic_verbalizer.far
INFO:NeMo-text-processing:Created cache/ar_tn_True_deterministic_verbalizer.far
 NeMo-text-processing :: DEBUG    :: tokens { cardinal { integer: "مئة وثلاثة وعشرون" } }
DEBUG:NeMo-text-processing:tokens { cardinal { integer: "مئة وثلاثة وعشرون" } }


مئة وثلاثة وعشرون


In [ ]:
from nemo_text_processing.text_normalization.normalize import Normalizer

normalizer = Normalizer(input_case="lower_cased", lang="en")

print(normalizer.normalize("01/01/2025", verbose=True))

 NeMo-text-processing :: INFO     :: Creating ClassifyFst grammars.
INFO:NeMo-text-processing:Creating ClassifyFst grammars.
 NeMo-text-processing :: DEBUG    :: cardinal:  0.37s -- 6247 nodes
DEBUG:NeMo-text-processing:cardinal:  0.37s -- 6247 nodes
 NeMo-text-processing :: DEBUG    :: ordinal:  0.56s -- 1478 nodes
DEBUG:NeMo-text-processing:ordinal:  0.56s -- 1478 nodes
 NeMo-text-processing :: DEBUG    :: decimal:  0.18s -- 3151 nodes
DEBUG:NeMo-text-processing:decimal:  0.18s -- 3151 nodes
 NeMo-text-processing :: DEBUG    :: fraction:  0.19s -- 4254 nodes
DEBUG:NeMo-text-processing:fraction:  0.19s -- 4254 nodes
 NeMo-text-processing :: DEBUG    :: measure:  6.71s -- 49514 nodes
DEBUG:NeMo-text-processing:measure:  6.71s -- 49514 nodes
 NeMo-text-processing :: DEBUG    :: date:  0.39s -- 4456 nodes
DEBUG:NeMo-text-processing:date:  0.39s -- 4456 nodes
 NeMo-text-processing :: DEBUG    :: time:  0.10s -- 418 nodes
DEBUG:NeMo-text-processing:time:  0.10s -- 418 nodes
 NeMo-text-proc

january first twenty twenty five


In [ ]:
import nemo_text_processing
print(nemo_text_processing.__file__)

/usr/local/lib/python3.12/dist-packages/nemo_text_processing/__init__.py


In [ ]:
%cd /usr/local/lib/python3.12/dist-packages/nemo_text_processing
!ls

/usr/local/lib/python3.12/dist-packages/nemo_text_processing
fst_alignment  __init__.py		   __pycache__
g2p	       inverse_text_normalization  text_normalization
hybrid	       package_info.py		   utils


In [ ]:
!find . -name "*.py"

./hybrid/utils.py
./hybrid/model_utils.py
./hybrid/wfst_lm_rescoring.py
./hybrid/mlm_scorer.py
./hybrid/__init__.py
./text_normalization/it/verbalizers/verbalize.py
./text_normalization/it/verbalizers/electronic.py
./text_normalization/it/verbalizers/decimal.py
./text_normalization/it/verbalizers/money.py
./text_normalization/it/verbalizers/cardinal.py
./text_normalization/it/verbalizers/measure.py
./text_normalization/it/verbalizers/verbalize_final.py
./text_normalization/it/verbalizers/__init__.py
./text_normalization/it/verbalizers/time.py
./text_normalization/it/utils.py
./text_normalization/it/data/electronic/__init__.py
./text_normalization/it/data/measure/__init__.py
./text_normalization/it/data/__init__.py
./text_normalization/it/data/numbers/__init__.py
./text_normalization/it/data/whitelist/__init__.py
./text_normalization/it/data/money/__init__.py
./text_normalization/it/taggers/whitelist.py
./text_normalization/it/taggers/word.py
./text_normalization/it/taggers/electronic.p

In [ ]:
%cd /usr/local/lib/python3.12/dist-packages/nemo_text_processing/text_normalization/ar

/usr/local/lib/python3.12/dist-packages/nemo_text_processing/text_normalization/ar


In [ ]:
%%bash
BASE_DIR="/usr/local/lib/python3.12/dist-packages/nemo_text_processing/text_normalization/darija"
mkdir -p $BASE_DIR/taggers
mkdir -p $BASE_DIR/verbalizers
touch $BASE_DIR/__init__.py
touch $BASE_DIR/taggers/__init__.py
touch $BASE_DIR/verbalizers/__init__.py

In [ ]:
# Installer les dépendances système d’OpenFst (déjà présentes)
!apt-get install -y libfst-dev

# Installer Pynini avec dépendances
!pip install pynini==2.1.6.post1

# Cloner la version complète de NeMo‑text‑processing
!git clone https://github.com/NVIDIA/NeMo-text-processing.git
%cd NeMo-text-processing

# Installer le package depuis la source
!pip install -e .

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libfst-dev is already the newest version (1.6.3-2ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 45 not upgraded.
Cloning into 'NeMo-text-processing'...
remote: Enumerating objects: 20181, done.
remote: Counting objects: 100% (322/322), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 20181 (delta 264), reused 193 (delta 193), pack-reused 19859 (from 2)
Receiving objects: 100% (20181/20181), 27.20 MiB | 16.89 MiB/s, done.
Resolving deltas: 100% (15371/15371), done.
/content/NeMo-text-processing/NeMo-text-processing
Obtaining file:///content/NeMo-text-processing/NeMo-text-processing
  Preparing metadata (setup.py) ... done
  Attempting uninstall: nemo_text_processing
    Found existing installation: nemo_text_processing 1.1.0
    Uninstalling nemo_text_processing-1.1.0:
      Successfully uninstalled nemo_text_processing-1.1.0
  Running setup.py develop for nem

In [ ]:
# Supposons que tu as ton dossier Darija dans dist-packages
!cp -r /usr/local/lib/python3.12/dist-packages/nemo_text_processing/text_normalization/darija \
       /content/NeMo-text-processing/nemo_text_processing/text_normalization/

In [ ]:
!ls /content/NeMo-text-processing/nemo_text_processing/text_normalization/darija

date  __init__.py  __pycache__


In [ ]:
# Supprime le deuxième dossier dupliqué
!rm -rf /content/NeMo-text-processing/NeMo-text-processing

# Vérifie ce qu'il reste
!ls /content/NeMo-text-processing

CHANGELOG.md	 LICENSE			reinstall.sh  tools
CONTRIBUTING.md  MANIFEST.in			requirements  tutorials
data		 nemo_text_processing		setup.cfg
__init__.py	 nemo_text_processing.egg-info	setup.py
Jenkinsfile	 README.md			tests


In [ ]:
import sys
sys.path.insert(0, '/content/NeMo-text-processing')

In [ ]:
import sys
sys.path.insert(0, '/content/NeMo-text-processing')

from nemo_text_processing.text_normalization.darija.date import DATE_GRAPH, VERBALIZE_DATE
print("Import OK")

ModuleNotFoundError: No module named 'nemo_text_processing.text_normalization.darija'

jai dit que on aura texte et audio on voit les aligner ;puis pour chaque mot du texte on va tirere ses phoneme depuis laudio et grace a lalignment ; cela doit etre fait pour tout les paire audio texte puis on comparer les phoneme si il se ressemnle si oui on recuepre les mots qui ont ce phoneme

In [ ]:
from huggingface_hub import login

login()  # une interface te demandera de coller ton token

In [ ]:
!pip install datasets transformers torchaudio scikit-learn

In [ ]:
import re
import torch
import numpy as np
from datasets import load_dataset
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
from sklearn.cluster import DBSCAN
from collections import defaultdict

# =========================
# 1. DATASETS CONFIG (auto colonne)
# =========================
DATASETS = {

    #"doda": {
    #    "path": "atlasia/DODa-audio-dataset",
     #   "text_candidates": ["darija_Arab_new", "darija_Arab_old", "sentence"]
    #},
    "majjodi": {
        "path": "abdeljalilELmajjodi/Darija_audio_text",
        "text_candidates": ["text", "sentence"]
    }#,
    #"mgb5": {
     #   "path": "ArabicSpeech/MGB-5",
      #  "text_candidates": ["text"]
    #},
    #"darija_speech": {
     #   "path": "adiren7/darija_speech_to_text",
      #  "text_candidates": ["sentence", "text"]
    #}
}

# =========================
# 2. MODEL (CTC ALIGNMENT)
# =========================
model_id = "jonatasgrosman/wav2vec2-large-xlsr-53-arabic"
processor = Wav2Vec2Processor.from_pretrained(model_id)
model = Wav2Vec2ForCTC.from_pretrained(model_id)
model.eval()

# =========================
# 3. TEXT FILTER
# =========================
def is_darija_arabic(text):
    if not isinstance(text, str):
        return False
    if re.search(r"[A-Za-z]", text):
        return False
    return bool(re.search(r"[\u0600-\u06FF]", text))

def clean(text):
    return re.sub(r"[^\u0600-\u06FF\s]", "", text).strip()

def pick_text(example, candidates):
    for c in candidates:
        if c in example:
            return example[c]
    return None

# =========================
# 4. EMBEDDING AUDIO (phonetic representation)
# =========================
def get_embedding(audio):
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt", padding=True)

    with torch.no_grad():
        outputs = model.wav2vec2(inputs.input_values)

    return outputs.last_hidden_state.mean(dim=1).squeeze().numpy()

# =========================
# 5. ALIGNMENT (CTC approximation correcte)
# =========================
def align_words(audio, text):
    words = text.split()

    inputs = processor(audio, sampling_rate=16000, return_tensors="pt")

    with torch.no_grad():
        logits = model(inputs.input_values).logits[0]

    T = logits.shape[0]
    duration = len(audio) / 16000

    step = T / max(len(words), 1)

    aligned = []

    for i, w in enumerate(words):
        start_f = int(i * step)
        end_f = int((i + 1) * step)

        start_t = start_f * (duration / T)
        end_t = end_f * (duration / T)

        aligned.append((w, start_t, end_t))

    return aligned

# =========================
# 6. LOAD ALL DATASETS
# =========================
word_vectors = []
word_texts = []

for name, cfg in DATASETS.items():

    print("Processing:", name)

    ds = load_dataset(cfg["path"], split="train", streaming=True)

    count = 0

    for ex in ds:

        text = pick_text(ex, cfg["text_candidates"])
        audio = ex["audio"]

        if text is None:
            continue

        if not is_darija_arabic(text):
            continue

        text = clean(text)

        if len(text.split()) < 2:
            continue

        audio_array = audio["array"]

        try:
            alignment = align_words(audio_array, text)
        except:
            continue

        for word, start, end in alignment:

            try:
                sr = 16000
                s = int(start * sr)
                e = int(end * sr)

                segment = audio_array[s:e]

                emb = get_embedding(segment)

                word_vectors.append(emb)
                word_texts.append(word)

            except:
                continue

        count += 1
        if count > 50:
            break

# =========================
# 7. CLUSTERING PHONETIC
# =========================
X = np.array(word_vectors)

clustering = DBSCAN(eps=0.35, min_samples=2, metric="cosine").fit(X)

labels = clustering.labels_

clusters = defaultdict(list)

for i, l in enumerate(labels):
    if l == -1:
        continue
    clusters[l].append(word_texts[i])

# =========================
# 8. DICTIONARY NORMALIZATION
# =========================
norm_dict = {}

for _, words in clusters.items():

    canonical = max(set(words), key=words.count)

    for w in words:
        if w != canonical:
            norm_dict[w] = canonical

# =========================
# 9. OUTPUT
# =========================
print("\n--- NORMALIZATION DICTIONARY ---\n")

for k, v in list(norm_dict.items())[:100]:
    print(k, "→", v)

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Processing: doda
Processing: majjodi


README.md:   0%|          | 0.00/551 [00:00<?, ?B/s]

Processing: mgb5
Processing: darija_speech


README.md:   0%|          | 0.00/778 [00:00<?, ?B/s]


--- NORMALIZATION DICTIONARY ---

هوما → من
مخبيين → من
شي → من
حاجة → من
انا → من
متيقن → من
باينة → من
كيحاولو → من
يبقاو → من
مبردين → من
لوطيلات → من
مبيناش → من
فيهم → من
مريحين → من
بزاف → من
غالبا → من
غيجريو → من
عليه → من
الخدمة → من
طبعا → من
راه → من
مكتئب → من
كيبان → من
ليا → من
غنمشي → من
ارا → من
داك → من
الصاك → من
كنت → من
ديما → من
عارف → من
انها → من
بغاتنا → من
نموتو → من
بغيت → من
نعرف → من
شحال → من
بقى → من
ديال → من
الوقت → من
باش → من
نقرا → من
غيكون → من
عندنا → من
امتحان → من
الى → من
هادشي → من
اللي → من
قصدتي → من
السيمانة → من
الجاية → من
نهار → من
الخميس → من
غنغطيو → من
كاع → من
المواد → من
تال → من
التلات → من
الجاي → من
ايوا → من
بدا → من
القراية → من
بكري → من
واخا → من
عندي → من
اي → من
سؤال → من
واش → من
نصيفطهم → من
لك → من
في → من
الإيمايل → من
ولكن → من
غنجاوب → من
عليهم → من
فأوقات → من
اذن → من
عافاك → من
ما → من
تصيفطهمش → من
معطل → من
لاربع → من
بالعشية → من
شكرا → من
الالة → من
اش → من
كيعني → من
كيفاش → من
كتابك → من
غيعاوني → من
نتهدن → م

In [ ]:
def nombre_vers_darija(n):
    """
    Convertit un entier en sa verbalisation en Darija marocaine (script arabe).
    Gère les nombres de 0 à 999 999.
    """
    if not isinstance(n, int):
        raise ValueError("L'entrée doit être un entier.")

    if n == 0:
        return "صفر"

    # Dictionnaires de base
    unites = ["", "واحد", "جوج", "تلاتة", "ربعة", "خمسة", "ستة", "سبعة", "تمنية", "تسعود"]
    dizaines_10_19 = ["عشرة", "حداش", "طناش", "تلطاش", "ربعطاش", "خمستاش", "سطاش", "سبعطاش", "تمنطاش", "تسعطاش"]
    dizaines = ["", "عشرة", "عشرين", "تلاتين", "ربعين", "خمسين", "ستين", "سبعين", "تمنين", "تسعين"]

    # Centaines fusionnées
    centaines = ["", "مية", "ميتين", "تلت مية", "ربع مية", "خمس مية", "ست مية", "سبع مية", "تمن مية", "تسع مية"]

    # NOUVEAU : Milliers fusionnés (de 3000 à 10000)
    milliers_fuses = {
        3: "تلتالاف", 4: "ربعالاف", 5: "خمسالاف", 6: "ستالاف",
        7: "سبعالاف", 8: "تمنالاف", 9: "تسعالاف", 10: "عشرالاف"
    }

    def traiter_dizaines(num):
        if num < 10:
            return unites[num]
        elif 10 <= num <= 19:
            return dizaines_10_19[num - 10]
        else:
            unite = num % 10
            dizaine = num // 10
            if unite == 0:
                return dizaines[dizaine]
            else:
                # CORRECTION : Remplacer "جوج" par "تنين" dans les nombres composés
                nom_unite = "تنين" if unite == 2 else unites[unite]
                return nom_unite + " و " + dizaines[dizaine]

    def traiter_centaines(num):
        if num < 100:
            return traiter_dizaines(num)
        else:
            centaine = num // 100
            reste = num % 100
            if reste == 0:
                return centaines[centaine]
            else:
                return centaines[centaine] + " و " + traiter_dizaines(reste)

    # Logique principale pour assembler le tout
    if n < 1000:
        return traiter_centaines(n).strip()

    # Traitement des milliers
    millier = n // 1000
    reste = n % 1000

    str_millier = ""
    if millier == 1:
        str_millier = "الف"
    elif millier == 2:
        str_millier = "الفين"
    elif 3 <= millier <= 10:
        # CORRECTION : Utilisation du dictionnaire des milliers fusionnés
        str_millier = milliers_fuses[millier]
    else:
        str_millier = traiter_centaines(millier) + " الف"

    if reste == 0:
        return str_millier.strip()
    else:
        return (str_millier + " و " + traiter_centaines(reste)).strip()

# --- TESTS ---
print(nombre_vers_darija(8570)) # Sortie attendue : تمنالاف و خمس مية
print(nombre_vers_darija(635))  # Sortie attendue : ست مية و تنين و ربعين
print(nombre_vers_darija(999))  # Sortie attendue : مية و جوج (Ici "jouj" est préservé car il n'est pas lié à une dizaine)

تمنالاف و خمس مية و سبعين
ست مية و خمسة و تلاتين
تسع مية و تسعود و تسعين
